In [ ]:
#!pip install -q -U langchain langchain-openai langchain-core langgraph python-dotenv

# Skills en LangChain: *progressive disclosure*

Referencias:
- https://docs.langchain.com/oss/python/langchain/multi-agent/skills
- https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant

Objetivos:

- Entender qué problema resuelven las skills.
- Definir una clase `Skill` y un catálogo de skills.
- Crear dos tools, `list_skills` y `load_skill`, para que el agente descubra y cargue skills bajo demanda.
- Ver al agente usar las skills para armar una **ficha de matrícula** y una **solicitud de certificado**.

## ¿Qué problema resuelven las skills?

Imagina que el agente debe saber hacer muchos trámites: matrícula, certificados, retiros, reclamos, etc. Cada uno tiene sus propios datos, validaciones y formato de salida.

**Opción ingenua:** poner todas las instrucciones en el `system_prompt`.
- El prompt crece con cada trámite nuevo (más tokens y más costo en **cada** llamada).
- El modelo se confunde con instrucciones que no aplican a la pregunta actual.
- Es difícil que distintos equipos mantengan sus instrucciones por separado.

**Patrón Skills (progressive disclosure):** el agente solo conoce un **resumen** de cada skill y carga el **contenido completo** cuando lo necesita.

| Capa | Qué contiene | Cuándo lo ve el agente |
|------|--------------|------------------------|
| Metadata | `name` + `description` (1-2 frases) | Al llamar a `list_skills` |
| Contenido | Instrucciones, validaciones, plantillas | Al llamar a `load_skill` |

Flujo:

```
Usuario: "Quiero matricularme"
   └─▶ Agente llama a list_skills()          → ve el resumen de las skills
   └─▶ Agente llama a load_skill("ficha_matricula") → recibe las instrucciones completas
   └─▶ Agente sigue las instrucciones y responde
```

Una skill **no es un sub-agente**: es solo texto (un prompt especializado) que entra a la conversación como resultado de una tool.

In [ ]:
import getpass
import os

from dotenv import load_dotenv

# Carga el .env que está a la misma altura que este notebook
load_dotenv(".env")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # cambia el modelo si lo necesitas

## 1. La clase `Skill`

Igual que en la documentación de LangChain, una skill es un `TypedDict` con tres campos:

- `name`: identificador único (lo usa el agente para cargarla).
- `description`: resumen corto; es lo único que el agente ve antes de cargarla, así que debe dejar claro **cuándo** usarla.
- `content`: las instrucciones completas.

In [ ]:
from typing import TypedDict


class Skill(TypedDict):
    """Skill que se revela al agente de forma progresiva."""

    name: str  # Identificador único de la skill
    description: str  # Descripción breve (1-2 frases) que se muestra al listar
    content: str  # Instrucciones completas que se cargan solo cuando se necesitan

## 2. El catálogo de skills

Definimos dos skills de la academia. Como este notebook no usa la base de conocimiento (Qdrant), cada skill trae su propio **catálogo de programas** dentro del `content`. En el API, en cambio, la skill le pide al agente consultar `retrieve_documents`.

In [ ]:
FICHA_MATRICULA = """# Ficha de matrícula

## Objetivo
Guiar al alumno para completar su ficha de matrícula y entregarle la ficha final
para su confirmación.

## Programas disponibles
| Programa                         | Duración  | Inversión | Modalidades              |
|----------------------------------|-----------|-----------|--------------------------|
| Agentes de IA                    | 8 semanas | S/ 1,200  | Virtual en vivo          |
| Data Engineering                 | 12 semanas| S/ 1,800  | Virtual en vivo, Híbrido |
| Machine Learning                 | 10 semanas| S/ 1,500  | Virtual en vivo          |

## Datos obligatorios
| Campo               | Validación                                              |
|---------------------|---------------------------------------------------------|
| Nombres             | Texto no vacío                                          |
| Apellidos           | Texto no vacío                                          |
| DNI                 | Exactamente 8 dígitos numéricos                         |
| Fecha de nacimiento | Formato DD/MM/AAAA; el alumno debe ser mayor de 18 años |
| Correo electrónico  | Formato válido (usuario@dominio.ext)                    |
| Celular             | 9 dígitos y debe empezar con 9                          |
| Programa            | Debe estar en la tabla de programas disponibles         |
| Modalidad           | Una de las modalidades del programa elegido             |
| Medio de pago       | Tarjeta, transferencia bancaria o Yape/Plin             |

## Pasos
1. Pide los datos que falten de forma conversacional. Nunca inventes datos del alumno.
2. Valida cada dato. Si alguno es inválido, explica el motivo y pídelo de nuevo.
3. Si el programa no está en la tabla, muestra los programas disponibles.
4. Con todos los datos válidos, muestra la ficha con la plantilla y pide confirmación.

## Plantilla de la ficha
```
FICHA DE MATRÍCULA
==================
Alumno:              <Nombres> <Apellidos>
DNI:                 <DNI>
Fecha de nacimiento: <DD/MM/AAAA>
Correo:              <correo>
Celular:             <celular>

Programa:            <programa>
Modalidad:           <modalidad>
Duración:            <duración>
Inversión:           <inversión>
Medio de pago:       <medio de pago>
Estado:              PENDIENTE DE CONFIRMACIÓN
```
"""

SOLICITUD_CERTIFICADO = """# Solicitud de certificado o constancia de estudios

## Tipos de documento
- Certificado de aprobación: solo si el alumno terminó y aprobó el programa
  (nota final mínima 14 y asistencia mínima 80%).
- Constancia de estudios: para alumnos que están cursando un programa.

## Datos obligatorios
| Campo               | Validación                                                    |
|---------------------|---------------------------------------------------------------|
| Nombres y apellidos | Texto no vacío                                                |
| DNI                 | Exactamente 8 dígitos numéricos                               |
| Correo electrónico  | Formato válido (usuario@dominio.ext)                          |
| Programa            | Agentes de IA, Data Engineering o Machine Learning            |
| Tipo de documento   | "certificado" o "constancia"                                  |
| Estado del programa | "culminado" (para certificado) o "en curso" (para constancia) |
| Formato             | "digital" (PDF) o "físico" (recojo en sede)                   |
| Motivo              | Texto breve                                                   |

## Pasos
1. Pregunta qué tipo de documento necesita, si no lo dijo.
2. Pide los datos que falten. Nunca inventes datos del alumno.
3. Si pide certificado pero su programa sigue en curso, explícale que solo puede
   solicitar una constancia y ofrécele esa opción.
4. Muestra la solicitud con la plantilla y pide confirmación.
5. Plazo de entrega: 5 días hábiles si es digital, 10 días hábiles si es físico.

## Plantilla de la solicitud
```
SOLICITUD DE <CERTIFICADO DE APROBACIÓN | CONSTANCIA DE ESTUDIOS>
=================================================================
Solicitante:         <Nombres y apellidos>
DNI:                 <DNI>
Correo:              <correo>
Programa:            <programa>
Estado del programa: <culminado | en curso>
Formato:             <digital | físico>
Motivo:              <motivo>
Plazo estimado:      <5 | 10> días hábiles
Estado:              PENDIENTE DE CONFIRMACIÓN
```
"""

SKILLS: list[Skill] = [
    {
        "name": "ficha_matricula",
        "description": (
            "Procedimiento para registrar la matrícula de un alumno en un programa "
            "de la academia: datos a solicitar, validaciones y formato de la ficha."
        ),
        "content": FICHA_MATRICULA,
    },
    {
        "name": "solicitud_certificado",
        "description": (
            "Procedimiento para que un alumno o egresado solicite un certificado de "
            "aprobación o una constancia de estudios: requisitos, datos y formato."
        ),
        "content": SOLICITUD_CERTIFICADO,
    },
]

## 3. Las tools: `list_skills` y `load_skill`

Son tools normales decoradas con `@tool`:

- `list_skills`: devuelve solo la **metadata** (nombre + descripción). Es barata en tokens.
- `load_skill`: devuelve el **contenido completo** de una skill. Si el nombre no existe, devuelve la lista de nombres válidos para que el agente pueda corregirse.

Recuerda que el *docstring* es la descripción que ve el modelo para decidir cuándo llamar cada tool.

In [ ]:
from langchain_core.tools import tool


@tool
def list_skills() -> str:
    """Lista las skills disponibles (nombre y descripción breve). Úsala para
    descubrir qué procedimientos especializados existen antes de cargar uno."""
    return "\n".join(f"- {skill['name']}: {skill['description']}" for skill in SKILLS)


@tool
def load_skill(skill_name: str) -> str:
    """Carga el contenido completo de una skill: instrucciones, validaciones y
    plantillas para atender un tipo de solicitud. Úsala después de list_skills
    con el nombre exacto de la skill (por ejemplo, "ficha_matricula")."""
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return f"Skill cargada: {skill_name}\n\n{skill['content']}"

    available = ", ".join(skill["name"] for skill in SKILLS)
    return f"La skill '{skill_name}' no existe. Skills disponibles: {available}"

Antes de dárselas al agente, probamos las tools directamente con `.invoke`:

In [ ]:
print(list_skills.invoke({}))

In [ ]:
print(load_skill.invoke({"skill_name": "ficha_matricula"})[:400], "...")

In [ ]:
print(load_skill.invoke({"skill_name": "reclamos"}))

### ¿Cuánto contexto ahorramos?

Comparamos lo que ocuparía el prompt si pusiéramos **todas** las skills completas contra lo que ve el agente al listar solo la metadata. Usamos caracteres como aproximación de tokens.

In [ ]:
todo_el_contenido = sum(len(skill["content"]) for skill in SKILLS)
solo_metadata = len(list_skills.invoke({}))

print(f"Todas las skills completas: {todo_el_contenido:>6} caracteres")
print(f"Solo metadata (list_skills): {solo_metadata:>6} caracteres")
print(f"Ahorro: {1 - solo_metadata / todo_el_contenido:.0%}")

Con 2 skills el ahorro ya es notorio; con 20 o 50 skills la diferencia es enorme, porque en cada pregunta solo se carga la skill que hace falta.

## 4. Creando el agente

El `system_prompt` no incluye las instrucciones de ningún trámite: solo le dice al agente **cómo** usar las skills.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

SYSTEM_PROMPT = (
    "Eres el asistente de una academia. Responde siempre en español y sé conciso. "
    "Tienes skills con procedimientos especializados de la academia. Cuando el usuario "
    "quiera hacer un trámite o generar un documento, primero usa list_skills para ver "
    "las skills disponibles, luego usa load_skill con la skill que corresponda y sigue "
    "sus instrucciones al pie de la letra. No inventes procedimientos que no estén en "
    "una skill."
)

llm = ChatOpenAI(model=OPENAI_MODEL)

agent = create_agent(
    model=llm,
    tools=[list_skills, load_skill],
    system_prompt=SYSTEM_PROMPT,
)

Definimos una función auxiliar para ver qué tools llamó el agente y su respuesta final, sin imprimir el contenido completo de cada skill.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage


def mostrar_traza(messages):
    """Imprime las tools que llamó el agente y la respuesta final."""
    for message in messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            for call in message.tool_calls:
                print(f"🔧 {call['name']}({call['args']})")
    print("\n🤖 Respuesta:\n")
    print(messages[-1].content)

## 5. Ficha de matrícula

El usuario solo dice que quiere matricularse. Esperamos ver `list_skills` y luego `load_skill("ficha_matricula")`; después, el agente pide los datos de la ficha.

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="Hola, quiero matricularme en un programa")]})

mostrar_traza(response["messages"])

Para ver el detalle completo (incluido el `ToolMessage` con el contenido de la skill), usamos `pretty_print` como en el notebook anterior:

In [ ]:
for message in response["messages"]:
    message.pretty_print()

### Continuando la conversación

Para continuar, le pasamos al agente **todos** los mensajes anteriores más el nuevo. Así la skill cargada sigue en el contexto y el agente no necesita volver a cargarla.

Enviamos los datos con un **DNI inválido** (7 dígitos) para ver que aplica las validaciones de la skill.

In [ ]:
messages = response["messages"] + [
    HumanMessage(
        content=(
            "Soy Ana Torres Ríos, DNI 4567123, nací el 15/03/1995, correo ana.torres@gmail.com, "
            "celular 987654321. Quiero llevar Agentes de IA, virtual en vivo, y pago con Yape."
        )
    )
]

response = agent.invoke({"messages": messages})

mostrar_traza(response["messages"][len(messages):])

El agente debería pedir corregir el DNI. Lo corregimos y esperamos la ficha final con la plantilla de la skill:

In [ ]:
messages = response["messages"] + [HumanMessage(content="Perdón, mi DNI es 45671238")]

response = agent.invoke({"messages": messages})

mostrar_traza(response["messages"][len(messages):])

> **Nota para el API:** en el API el historial que se reenvía al agente solo guarda preguntas y respuestas (no los `ToolMessage`). Por eso, allí el `system_prompt` le pide al agente **volver a cargar la skill** en cada turno de un trámite en curso.

## 6. Solicitud de certificado

Con el mismo agente, ahora pedimos un certificado. El agente debe elegir **otra** skill. Además, el programa sigue en curso, así que la skill indica ofrecer una constancia en lugar del certificado.

In [ ]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Necesito mi certificado del programa Data Engineering, todavía lo estoy "
                    "cursando. Soy Luis Paredes, DNI 70123456, luis.paredes@outlook.com. "
                    "Lo quiero digital para presentarlo en mi trabajo."
                )
            )
        ]
    }
)

mostrar_traza(response["messages"])

## 7. Preguntas que no necesitan skills

Si la pregunta no es un trámite, el agente no debería cargar ninguna skill (y así no gasta tokens en ellas).

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="¿Qué es un agente de IA? Respóndeme en una frase.")]})

mostrar_traza(response["messages"])

## 8. Agregar una skill nueva

Para agregar una skill basta con sumarla a `SKILLS`. **No cambian las tools ni el `system_prompt`**: el agente la descubre con `list_skills`.

Agregamos una skill de **solicitud de retiro** de un programa.

In [ ]:
SKILLS.append(
    {
        "name": "solicitud_retiro",
        "description": (
            "Procedimiento para que un alumno se retire de un programa en curso: "
            "datos, política de devolución y formato de la solicitud."
        ),
        "content": """# Solicitud de retiro

## Política de devolución
- Retiro antes de la 2da semana: devolución del 80% de lo pagado.
- Retiro desde la 2da semana: no hay devolución; puede congelar y retomar en la
  siguiente edición sin costo adicional.

## Datos obligatorios
- Nombres y apellidos, DNI (8 dígitos), correo, programa, semana actual del programa
  y motivo del retiro.

## Pasos
1. Pide los datos que falten. Nunca inventes datos del alumno.
2. Explica la política que le aplica según la semana actual.
3. Si no le corresponde devolución, ofrécele congelar antes de confirmar el retiro.
4. Muestra la solicitud y pide confirmación.

## Plantilla
```
SOLICITUD DE RETIRO
===================
Alumno:          <Nombres y apellidos>
DNI:             <DNI>
Correo:          <correo>
Programa:        <programa>
Semana actual:   <semana>
Motivo:          <motivo>
Devolución:      <80% | No aplica>
Estado:          PENDIENTE DE CONFIRMACIÓN
```
""",
    }
)

print(list_skills.invoke({}))

Como las tools leen `SKILLS` en cada llamada, el mismo agente ya puede usar la nueva skill sin volver a crearlo:

In [ ]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Quiero retirarme de Machine Learning, estoy en la semana 3 porque me cambiaron de horario en el trabajo."
            )
        ]
    }
)

mostrar_traza(response["messages"])

## Resumen

- Una **skill** es un prompt especializado (`name`, `description`, `content`) que el agente carga bajo demanda.
- Con `list_skills` el agente ve solo la metadata; con `load_skill` recibe el contenido completo como un `ToolMessage`.
- El `system_prompt` se mantiene corto: solo explica cómo usar las skills.
- Agregar una skill es agregar un elemento a `SKILLS`, sin tocar las tools ni el prompt.

**En el API** (`03_agent_skills`), esto mismo vive en `src/services/skills.py`, las tools se agregan a `LOCAL_TOOLS` en `src/services/tools.py` y el `system_prompt` de `src/services/prompts.py` explica cómo usarlas, junto con las demás tools (retrieval, MCP de Neon).